# Beckman Coulter Biomek i7

The Biomek i7 is a liquid handler with two pods on one X rail: on this instrument a 96-channel
head (the Multichannel) on the left and a Span-8 on the right, each with a gripper. PyLabRobot
drives its boards directly over USB, with no Beckman software in the loop: it plans each move as
Biomek's planner does, checks it against the deck before sending it, and keeps the two pods apart.

This page runs on a simulated instrument, so it needs no hardware.

## Set up

`BiomekI7(simulation=True)` builds a simulated instrument with the package's generic configuration.
Labware is Biomek's own (`pylabrobot.resources.beckman_coulter`), placed by class name and deck
position as in a Biomek method's Instrument Setup.

In [ ]:
from pylabrobot.beckman_coulter.biomek_i7.device import BiomekI7
from pylabrobot.beckman_coulter.biomek_i7.layouts import place_labware
from pylabrobot.resources import set_tip_tracking, set_volume_tracking

set_tip_tracking(True)
set_volume_tracking(True)

biomek = BiomekI7(simulation=True)
deck = biomek.deck
place_labware(deck, [
  {"position": "TL4", "class": "BC230", "name": "tips_96"},     # the 96-head loads tips only on a tip-loading position
  {"position": "P5", "class": "BC230", "name": "tips_span8"},
  {"position": "P8", "class": "ThermoNUNC_350CB", "name": "reservoir"},
  {"position": "P6", "class": "BCFlat96", "name": "plate"},
])
tips_96, tips_span8 = deck.get_resource("tips_96"), deck.get_resource("tips_span8")
reservoir, plate = deck.get_resource("reservoir"), deck.get_resource("plate")
reservoir.get_item(0).set_volume(50_000.0)

In [ ]:
await biomek.setup()
head96, pipettes = biomek.head96, biomek.pipettes

## The 96-channel head

A transfer from the reservoir to every well of the plate, the tips back into their box.

In [ ]:
await head96.pick_up_tips(tips_96)
await head96.aspirate(reservoir, 150.0, liquid_height=2.0)
await head96.dispense(plate, 150.0, liquid_height=1.0)
await head96.drop_tips(tips_96)
plate.get_well("A1").tracker.get_used_volume()

## The Span-8

Eight probes on one X, spread evenly in Y. First its pumps are primed: their lines filled from the
system liquid, which also homes them.

In [ ]:
await pipettes.prime(2)

The left pod stands in the way of the Span-8's tips and plate: before the right pod moves there,
the left one is moved aside just far enough, as Biomek does.

In [ ]:
channels = list(range(8))
await pipettes.pick_up_tips(tips_span8["A1:H1"], use_channels=channels)
await pipettes.aspirate(plate["A1:H1"], [50.0] * 8, use_channels=channels, liquid_heights=[1.0] * 8)
await pipettes.dispense(plate["A7:H7"], [50.0] * 8, use_channels=channels, liquid_heights=[2.0] * 8)
await pipettes.discard_tips()
[well.tracker.get_used_volume() for well in plate["A7:H7"]]

## The gripper

Each pod has a gripper. The right one moves the plate from P6 to P9.

In [ ]:
gripper = biomek.right_x_arm.gripper
await gripper.pick_up_resource(plate)
await gripper.place_resource(deck.get_position("P9"))

In [ ]:
await biomek.park()
await biomek.stop()

## On an instrument

The driver needs the instrument's own configuration, made from its Biomek files: the instrument
file (`.bif`) and the Master Controller's instrument record. It is made once and checked against
the instrument at every setup:

```python
from pylabrobot.beckman_coulter.biomek_i7.driver.configuration import DeviceConfiguration

configuration = DeviceConfiguration.from_biomek(
  r"C:\Users\Public\Documents\Biomek5\Biomek_i7.bif",
  master_controller_db=r"C:\ProgramData\Beckman Coulter\Biomek Master Controller\Log.db3",
)
configuration.save("biomek_i7_configuration.json")

biomek = BiomekI7(declared_configuration_json="biomek_i7_configuration.json")
```

Only one program can hold the instrument's boards: stop the Biomek Master Controller service before
`setup()`. Before anything moves, clear the deck of everything but the method's labware, make sure
the light curtain works, and have someone at the emergency stop. `stop()` moves nothing: it rests
the arms by tripping the interlock.